# DuckDB SQL

In [1]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
minio_user = os.getenv("MINIO_ROOT_USER", "minioadmin")
minio_password = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET IF NOT EXISTS (
        TYPE S3,
        KEY_ID '{minio_user}',
        SECRET '{minio_password}',
        ENDPOINT 'localhost:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")
print("Connected to DuckDB and MinIO successfully")

Connected to DuckDB and MinIO successfully


In [ ]:
# query ot count total no of rows in whole file
q1 = """
SELECT COUNT(*) as total_rows from read_parquet('s3://analytics-data/ml_features.parquet')"""
res1 = con.execute(q1).df()
display(res1)

,total_rows
0,515268


In [ ]:
# query to describe the cols in the file 
q2 = """ DESCRIBE SELECT * FROM read_parquet('s3://analytics-data/ml_features.parquet')  """
res2 = con.execute(q2).df()
display(res2)

,column_name,column_type,null,key,default,extra
0,asset_symbol,VARCHAR,YES,None,None,None
1,asset_class,VARCHAR,YES,None,None,None
2,exchange,VARCHAR,YES,None,None,None
3,interval,VARCHAR,YES,None,None,None
4,date,TIMESTAMP,YES,None,None,None
5,open,DOUBLE,YES,None,None,None
6,high,DOUBLE,YES,None,None,None
7,low,DOUBLE,YES,None,None,None
8,close,DOUBLE,YES,None,None,None
9,volume,DOUBLE,YES,None,None,None


In [7]:
# query to read parquet from ml_features where to read specific asset on specific interval 
q3 = """
SELECT date, asset_symbol, open, high, low, close, volume
FROM read_parquet('s3://analytics-data/ml_features.parquet')
WHERE asset_symbol = 'AAPL' AND interval = '1h'
ORDER BY date ASC
LIMIT 5;
"""
res3 = con.execute(q3).df()
display(res3)

,date,asset_symbol,open,high,low,close,volume
0,2024-05-13 16:30:00,AAPL,186.289993,186.929993,186.250000,186.658600,8098301.0
1,2024-05-13 17:30:00,AAPL,186.660004,186.949997,186.369995,186.650101,5453676.0
2,2024-05-13 18:30:00,AAPL,186.659897,187.100006,186.559998,186.989899,6516486.0
3,2024-05-13 19:30:00,AAPL,186.985001,187.029999,186.070007,186.285004,9250536.0
4,2024-05-14 13:30:00,AAPL,187.649994,188.300003,186.550003,187.009995,14754549.0


In [ ]:
#query to get total no of rows in asset aapl for specific interval 
q4 = """
SELECT count(*) as total_rows 
FROM read_parquet('s3://analytics-data/ml_features.parquet') 
WHERE 
    asset_symbol = 'AAPL'
    AND interval = '1h';
"""
res4 = con.execute(q4).df()
display(res4)

,total_rows
0,3193


In [ ]:
# query to 